In [4]:
import numpy as np
import numpy as np
import torch
from scipy import signal
from trainSkeletonMorphing import Normalize

import matplotlib.pyplot as plt

In [10]:
def load_train_test_all(data_folder: str, pars=np.arange(10, 27)):
    """
    Method to load all training and test data from participants [pars]
    :param data_folder:
    :param pars:
    :return:
    """

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    scaler =  Normalize()
    for i in pars:
        if i == 13:
            continue
        #print(f'{data_folder}/morph_dataset/par_{i}_mediapipe_dataset.pth')


        par_dataset = torch.load(f'{data_folder}/morph_dataset/par_{i}_mediapipe_dataset.pth',
                                 map_location=torch.device(device))


        for d in par_dataset.datasets:
            d.par = i
            scaler.add_key_from_vector(d.csv_data, f"pose_gt")

        for d in par_dataset.datasets:
            d.csv_data = scaler.scale(d.csv_data,  f"pose_gt")
            d.align_procrustes()
            #scaler.add_key_from_vector(d.pose_inf, f"pose_inf")
            for camera in range(6):
                scaler.add_key_from_vector(d.pose_inf[:, camera], f"pose_inf_{camera}")


        for d in par_dataset.datasets:
            for camera in range(6):
                d.pose_inf[:, camera] = scaler.scale(d.pose_inf, f"pose_inf_{camera}")
            #d.pose_inf = scaler.scale(d.pose_inf,  f"pose_inf")




    return par_dataset, scaler
datapath = "E:\MoCap"
dataset, scaler = load_train_test_all(datapath, [14])
print(len(dataset))

ValueError: could not broadcast input array from shape (1700,6,16,3) into shape (1700,16,3)

In [6]:
import plotly.graph_objects as go
def plot_3d_keypoints(keypoints, model_name, wandb_name, epoch, fold_id):

    if wandb_name == 'morphed':
        color = "red"
        name = "Morphed KeyPoints"
        # wandb.log({'Test': fig})
    elif wandb_name == 'ground_truth':
        color = "green"
        name = "Ground Truth KeyPoints"

    elif wandb_name == 'hpe_truth':
        color = "blue"
        name = "HPE Truth KeyPoints"

    else:
        raise Exception("Invalid wandb name")



    colors = ['red', 'green', 'blue']
    # Define connections between related keypoints
    if model_name == 'mediapipe':
        connections = [(0, 1), (0, 2), (1, 3), (2, 4), (3, 5), (6, 7), (0, 6),
                        (1, 7), (6, 8), (7, 9), (8, 10), (9, 11), (10, 12),
                        (11, 13), (10, 14), (11, 15), (12, 14), (13, 15)]

    # Create a Plotly 3D scatter plot
    fig = go.Figure()

    # Extract X, Y, and Z coordinates from keypoints
    x, z, y = zip(*keypoints)

    # Scatter plot for keypoints
    fig.add_trace(go.Scatter3d(x=x, y=y, z=z, mode='markers', marker=dict(color=color, size=5)))

    # Plot connections
    for connection in connections:
        x_vals = [x[connection[0]], x[connection[1]]]
        y_vals = [y[connection[0]], y[connection[1]]]
        z_vals = [z[connection[0]], z[connection[1]]]
        fig.add_trace(go.Scatter3d(x=x_vals, y=y_vals, z=z_vals, mode='lines', line=dict(color=color)))

    # Combine x, y, z values into a single list
    #all_values = x + y + z

    # Find the minimum and maximum values
    min_value = np.min(keypoints) - abs(np.min(keypoints) * 0.1)
    max_value = np.max(keypoints) + abs(np.max(keypoints) * 0.1)

    # Update layout to set axis limits
    fig.update_layout(
        scene=dict(
            aspectmode='cube',
            xaxis=dict(title='X', range=[min_value, max_value]),
            yaxis=dict(title='Y', range=[min_value, max_value]),
            zaxis=dict(title='Z', range=[min_value, max_value])
        )
    )

    fig.show()



In [11]:
pose_gt_batch = dataset[0]['pose_gt']
pose_inf_batch = dataset[0]['pose_inf'][3]
plot_3d_keypoints(pose_gt_batch, 'mediapipe', 'morphed', 0, 0)
print(pose_gt_batch)

[[0.62028787 0.82533546 0.27394529]
 [0.62233089 0.82665791 0.16433353]
 [0.6068271  0.75065185 0.29318755]
 [0.60880043 0.75123027 0.144411  ]
 [0.63484863 0.67929597 0.284253  ]
 [0.63028005 0.67927339 0.1514484 ]
 [0.63290438 0.68818201 0.24506612]
 [0.6328724  0.68812381 0.19612704]
 [0.62831786 0.57906673 0.24977818]
 [0.62555165 0.5803381  0.1883086 ]
 [0.60342909 0.46203361 0.25160805]
 [0.60525906 0.4614247  0.18995271]
 [0.59415391 0.46347353 0.25047455]
 [0.59304407 0.46299785 0.18988505]
 [0.63339969 0.45948488 0.25354292]
 [0.63304688 0.45896265 0.18926911]]


In [12]:
pose_gt_batch = dataset[0]['pose_gt'].copy()
pose_inf_batch = dataset[0]['pose_inf'][2].copy()
cpy = pose_inf_batch.copy()
pose_inf_batch[:,1] = cpy[:, 2]
pose_inf_batch[:,2] = cpy[:, 1]

plot_3d_keypoints(pose_inf_batch, 'mediapipe', 'morphed', 0, 0)
print(pose_inf_batch)

[[0.67855594 0.34425151 0.68991386]
 [0.66307247 0.41389655 0.66618193]
 [0.6770152  0.34707535 0.74350674]
 [0.65553993 0.44177712 0.71216476]
 [0.69858519 0.37228578 0.78474545]
 [0.67523303 0.45614515 0.7577845 ]
 [0.69076438 0.39496075 0.77111461]
 [0.68326165 0.42608231 0.76103714]
 [0.69497484 0.41586134 0.84298456]
 [0.6836554  0.45430677 0.82974715]
 [0.68570167 0.43781545 0.92151212]
 [0.6775268  0.47737906 0.90896422]
 [0.67926635 0.43704598 0.92132818]
 [0.66928913 0.47553218 0.90922516]
 [0.70610546 0.44092832 0.92039154]
 [0.69607159 0.48186248 0.90747685]]


In [29]:
from skeletonMorphing.trainSkeletonMorphing import EveryNthSampler
import tqdm
exit()
from torch.utils.data import DataLoader, SubsetRandomSampler

column_mapping = {
    'RShoulder': 'RSJC', # 12 - 0
    'LShoulder': 'LSJC', # 11 - 1
    'RElbow': 'REJC', # 14 - 2
    'LElbow': 'LEJC', # 13 - 3
    'RWrist': 'RWJC', # 16 - 4
    'LWrist': 'LWJC', # 15 - 5
    'RHip': 'RHJC', # 24 - 6
    'LHip': 'LHJC', # 23 - 7
    'RKnee': 'RKJC', # 26 - 8
    'LKnee': 'LKJC', # 25  - 9
    'RAnkle': 'RAJC', # 28 - 10
    'LAnkle': 'LAJC', # 27 - 11
    'RHeel': 'RHEE', # 30 - 12
    'LHeel': 'LHEE', # 29 - 13
    'RFootIndex': 'RTOE', # 32 - 14
    'LFootIndex': 'LTOE', # 31 - 15
}
maps = list(column_mapping.keys())


def get_graph(datapath, shuffle):
    data = []
    for par in [12, 14, 15, 16]:
        dataset = load_train_test_all(datapath, [par])
        #sampler = EveryNthSampler(dataset, 25)
        #shuffled_sampler = sampler
        #shuffled_sampler = SubsetRandomSampler(list(sampler))
        dataloader = DataLoader(dataset, batch_size=32)
        #print(len(shuffled_sampler))
        for step, batch in enumerate(dataloader):
            for t in range(6):
                for j in range(16):
                    # Access data for each batch
                    #print(batch)
                    pose_gt_batch = batch['pose_gt']
                    pose_inf_batch = batch['pose_inf']
                    #print(pose_gt_batch.shape)
                    #print(pose_inf_batch.shape)
                    conf_inf = batch['confidences_inf']
                    sig = torch.norm(pose_inf_batch, dim=3)
                    #print(sig.shape)
                    sig = sig[:, t, j]

                    sig_noise = torch.norm(pose_gt_batch, dim=2)
                    #print(sig_noise.shape)
                    sig_noise = sig_noise[:, j]

                    corr = signal.correlate(sig_noise, sig)

                    lags = signal.correlation_lags(len(sig), len(sig_noise))

                    corr /= np.max(corr)
                    data.append({
                        'Participant': par,
                        'Camera': t,
                        'Joint': maps[j],
                        'Max': np.max(corr),
                        'Lag Index': lags[np.argmax(corr)],
                        'Batch' : step,
                        'n datasets' : len(dataset.datasets)
                    })
                    #print("MAX", np.max(corr), "Index", lags[np.argmax(corr)])

            #fig, (ax_orig, ax_noise, ax_corr) = plt.subplots(3, 1, figsize=(4.8, 4.8))

            #ax_orig.plot(sig)
            #ax_orig.set_title('Original signal')
            #ax_orig.set_xlabel('Sample Number')
            #ax_noise.plot(sig_noise)
            #ax_noise.set_title('Signal with noise')
            #ax_noise.set_xlabel('Sample Number')
            #ax_corr.plot(lags, corr)
            #ax_corr.set_title('Cross-correlated signal')
            #ax_corr.set_xlabel('Lag')
            #ax_orig.margins(0, 0.1)

            #ax_noise.margins(0, 0.1)

            #ax_corr.margins(0, 0.1)

            #fig.tight_layout()

            #plt.show()
    return data

data = get_graph(datapath, False)

TypeError: default_collate: batch must contain tensors, numpy arrays, numbers, dicts or lists; found <class 'skeletonMorphing.readDatasetMorph.ReadDatasetFiles'>

In [ ]:
from skeletonMorphing.trainSkeletonMorphing import Normalize


def load_train_test_all(data_folder: str, pars=np.arange(10, 27)):
    """
    Method to load all training and test data from participants [pars]
    :param data_folder:
    :param pars:
    :return:
    """
    scaler_train = Normalize()
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    train_dataset = None
    for i in pars:
        if i == 13:
            continue
        print(f'{data_folder}/morph_dataset/par_{i}_mediapipe_dataset.pth')


        par_dataset = torch.load(f'{data_folder}/morph_dataset/par_{i}_mediapipe_dataset.pth',
                                 map_location=torch.device(device))

        if train_dataset is None:
            #train_dataset, test_dataset = par_dataset.get_train_test()

            for d in par_dataset.datasets:

                scaler_train.add_key_from_vector(d.csv_data, f"pose_gt_{i}")
            # scaler_test = scaler_train
            for _, d in enumerate(par_dataset.datasets):
                if d.csv_data.size == 0:
                    continue

                par_dataset.datasets[_].csv_data = scaler_train.scale(d.csv_data,  f"pose_gt_{i}")
                par_dataset.datasets[_].par = i

            train_dataset = par_dataset
        else:

            for d in par_dataset.datasets:

                scaler_train.add_key_from_vector(d.csv_data, f"pose_gt_{i}")

            for _, d in enumerate(par_dataset.datasets):
                if d.csv_data.size == 0:
                    continue

                par_dataset.datasets[_].csv_data = scaler_train.scale(d.csv_data,  f"pose_gt_{i}")
                par_dataset.datasets[_].par = i

            train_dataset = torch.utils.data.ConcatDataset([train_dataset, par_dataset])



    print(len(train_dataset))

    #scaler.save("standardizer")
    return train_dataset

def load_train_test_all(data_folder: str, pars=np.arange(10, 27)):
    """
    Method to load all training and test data from participants [pars]
    :param data_folder:
    :param pars:
    :return:
    """

    scaler_train = Normalize()
    #scaler_test = Normalize()
    scaler_test = scaler_train
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    train_dataset = None
    test_dataset = None
    for i in pars:
        if i == 13:
            continue
        print(f'{data_folder}/morph_dataset/par_{i}_mediapipe_dataset.pth')


        par_dataset = torch.load(f'{data_folder}/morph_dataset/par_{i}_mediapipe_dataset.pth',
                                 map_location=torch.device(device))

        if train_dataset is None:
            train_dataset, test_dataset = par_dataset.get_train_test()
            print(train_dataset.datasets[0].csv_data.shape)
            print(train_dataset.datasets[0].pose_inf.shape)
            for d in train_dataset.datasets:

                scaler_train.add_key_from_vector(d.csv_data, f"pose_gt_{i}")
                #scaler_train.add_key_from_vector(i.pose_inf, "pose_inf")

            print(test_dataset)
            for d in test_dataset.datasets:
                scaler_test.add_key_from_vector(d.csv_data, f"pose_gt_{i}")
                #scaler_test.add_key_from_vector(i.pose_inf, "pose_inf")

            # scaler_test = scaler_train
            for _, d in enumerate(train_dataset.datasets):
                if d.csv_data.size == 0:
                    continue

                train_dataset.datasets[_].csv_data = scaler_train.scale(d.csv_data,  f"pose_gt_{i}")
                train_dataset.datasets[_].par = i

            for _, d in enumerate(test_dataset.datasets):
                if d.csv_data.size == 0:
                    continue
                test_dataset.datasets[_].csv_data = scaler_test.scale(d.csv_data,  f"pose_gt_{i}")
                test_dataset.datasets[_].par = i


        else:
            train, test = par_dataset.get_train_test()
            for d in train.datasets:
                scaler_train.add_key_from_vector(d.csv_data, f"pose_gt_{i}")
                #scaler_train.add_key_from_vector(i.pose_inf, "pose_inf")

            for d in test.datasets:
                scaler_test.add_key_from_vector(d.csv_data, f"pose_gt_{i}")

            for _, d in enumerate(train.datasets):
                if d.csv_data.size == 0:
                    continue

                train.datasets[_].csv_data = scaler_train.scale(d.csv_data,  f"pose_gt_{i}")
                train.datasets[_].par = i

            for _, d in enumerate(test.datasets):
                if d.csv_data.size == 0:
                    continue
                test.datasets[_].csv_data = scaler_test.scale(d.csv_data,  f"pose_gt_{i}")
                test.datasets[_].par = i

            train_dataset = torch.utils.data.ConcatDataset([train_dataset, train])
            test_dataset = torch.utils.data.ConcatDataset([test_dataset, test])

        print("NORM RESULTS")
        print(scaler_train.dict[f'pose_gt_{i}'])
        print(scaler_test.dict[f'pose_gt_{i}'])


    print(len(train_dataset))

    #scaler.save("standardizer")
    return train_dataset, test_dataset, scaler_train, scaler_test

def get_graph_2(dataset):
    data = []


    #sampler = EveryNthSampler(dataset, 25)
    #shuffled_sampler = sampler
    #shuffled_sampler = SubsetRandomSampler(list(sampler))
    dataloader = DataLoader(dataset, batch_size=64, shuffle=False)
    #print(len(shuffled_sampler)
    for step, batch in enumerate(dataloader):
        for t in range(6):
            for j in range(16):
                # Access data for each batch
                #print(batch)
                pose_gt_batch = batch['pose_gt']
                pose_inf_batch = batch['pose_inf']
                par = batch['par']

                #print(pose_gt_batch.shape)
                #print(pose_inf_batch.shape)

                sig = torch.norm(pose_inf_batch, dim=3)
                #print(sig.shape)
                sig = sig[:, t, j]

                sig_noise = torch.norm(pose_gt_batch, dim=2)
                #print(sig_noise.shape)
                sig_noise = sig_noise[:, j]

                corr = signal.correlate(sig_noise, sig)

                lags = signal.correlation_lags(len(sig), len(sig_noise))

                corr /= np.max(corr)
                data.append({
                    'Participant': par,
                    'Camera': t,
                    'Joint': maps[j],
                    'Max': np.max(corr),
                    'Lag Index': lags[np.argmax(corr)],
                    'Batch' : step,
                    'n datasets' : len(dataset.datasets)
                })

    return data

#data = get_graph_2(load_train_test_all(datapath, ['12'])[1])

In [ ]:
import pandas as pd
df = pd.DataFrame(data)
df

In [ ]:
df[df['Lag Index'] != 0]

In [ ]:
df[df['Lag Index'] != 0].groupby("Camera").count()

In [ ]:
df[df['Lag Index'] != 0].groupby("Lag Index").count()

In [ ]:

raise ExceptionGroup()

In [ ]:

for group_name, group_data in df.groupby([ "Joint", "Camera"]):
    print("Group:", group_name)
    print(group_data)

In [ ]:


column_mapping = {
    'RShoulder': 'RSJC', # 12 - 0
    'LShoulder': 'LSJC', # 11 - 1
    'RElbow': 'REJC', # 14 - 2
    'LElbow': 'LEJC', # 13 - 3
    'RWrist': 'RWJC', # 16 - 4
    'LWrist': 'LWJC', # 15 - 5
    'RHip': 'RHJC', # 24 - 6
    'LHip': 'LHJC', # 23 - 7
    'RKnee': 'RKJC', # 26 - 8
    'LKnee': 'LKJC', # 25  - 9
    'RAnkle': 'RAJC', # 28 - 10
    'LAnkle': 'LAJC', # 27 - 11
    'RHeel': 'RHEE', # 30 - 12
    'LHeel': 'LHEE', # 29 - 13
    'RFootIndex': 'RTOE', # 32 - 14
    'LFootIndex': 'LTOE', # 31 - 15
}
maps = list(column_mapping.keys())

def gransk_nøyere(batch, cam, joint):
    pose_gt_batch = batch['pose_gt']
    pose_inf_batch = batch['pose_inf']
    #sig = torch.norm(pose_inf_batch, dim=3)
    print("Cam", cam, "Joint", maps[joint],  "Gransker nøyere for XYZ")
    for i in range(3):
        sig = pose_inf_batch[:, cam, joint,i]

        #sig_noise = torch.norm(pose_gt_batch, dim=2)
        sig_noise = pose_gt_batch[:, joint,i]

        corr = signal.correlate(sig_noise, sig)

        lags = signal.correlation_lags(len(sig), len(sig_noise))

        corr /= np.max(corr)


        print(i, np.max(corr), lags[np.argmax(corr)])
        print("___________________________________________________________")




def get_graph(dataset, cam, joint):

    dataloader = DataLoader(dataset, batch_size=10000, shuffle=True)
    l = []
    c = []
    for step, batch in enumerate(dataloader):
        # Access data for each batch
        #print(batch)

        pose_gt_batch = batch['pose_gt']
        pose_inf_batch = batch['pose_inf']

        sig = torch.norm(pose_inf_batch, dim=3)
        sig = sig[:, cam, joint]

        sig_noise = torch.norm(pose_gt_batch, dim=2)
        sig_noise = sig_noise[:, joint]

        corr = signal.correlate(sig_noise, sig)

        lags = signal.correlation_lags(len(sig), len(sig_noise))

        corr /= np.max(corr)

        c.append(np.max(corr))
        l.append(lags[np.argmax(corr)])
        x = 0
        y = 0
        z = 0
        axes = [x, y, z]
        for i, axis in enumerate(axes):
            sig = pose_inf_batch[:, cam, joint, i]

            sig_noise = pose_gt_batch[:, joint, i]

            corr = signal.correlate(sig_noise, sig)

            lags = signal.correlation_lags(len(sig), len(sig_noise))

            corr /= np.max(corr)

            axes[i] = lags[np.argmax(corr)]

        #if lags[np.argmax(corr)] != 0:
            #gransk_nøyere(batch, cam, joint)

    #print("Participant:", par, " Camera", camera, " joint", maps[joint])
    #print("MAX", np.mean(c), "Index", np.mean(l))
    #print("___________________________________________________________")
    return np.mean(c), np.mean(l), axes[0], axes[1], axes[2]

data = []
for i in tqdm.tqdm(range(12, 17)):
    if i == 13:
        continue
    dataset = load_train_test_all(datapath, [i])


    print("Participant: ", i)
    for cam in range(6):
        print("Cam: ", cam)
        for j in range(16):
            c, l, x, y, z = get_graph(dataset, cam, j)
            if l != 0:
                #print("Participant:", i, " Camera", cam, " joint", maps[j])
                #print("MAX", c, "Index", l)
                #print("___________________________________________________________")
                data.append({
                    'Participant': i,
                    'Camera': cam,
                    'Joint': maps[j],
                    'Max': c,
                    'Index': l,
                    'X' : x,
                    'Y' : y,
                    'Z' : z
                })

#df = pd.DataFrame(data)



In [ ]:
df = pd.DataFrame(data)
df

In [ ]:
data_2 = []

def load_train_test_all(data_folder: str, pars=np.arange(10, 27), val = False):
    """
    Method to load all training and test data from participants [pars]
    :param data_folder:
    :param pars:
    :return:
    """

    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    for i in pars:
        if i == 13:
            continue
        #print(f'{data_folder}/morph_dataset/par_{i}_mediapipe_dataset.pth')


        par_dataset = torch.load(f'{data_folder}/morph_dataset/par_{i}_mediapipe_dataset.pth',
                                 map_location=torch.device(device)).get_train_test()[int(val)]


        for d in par_dataset.datasets:
            d.par = i

    return par_dataset
datapath = "E:\MoCap"
dataset = load_train_test_all(datapath, [12])
print(len(dataset))
#### Only test Validation
for i in tqdm.tqdm(range(12, 17)):
    for val in range(2):

        if i == 13:
            continue
        dataset = load_train_test_all(datapath, [i], val)

        print("Participant: ", i)
        for cam in range(6):
            print("Cam: ", cam)
            for j in range(16):
                c, l, x, y, z = get_graph(dataset, cam, j)

                data_2.append({
                        'Participant': i,
                        'Camera': cam,
                        'Joint': maps[j],
                        'Max': c,
                        'Index': l,
                        'X' : x,
                        'Y' : y,
                        'Z' : z,
                        'Validation_set' : bool(val)
                    })
                #if l != 0:
                    #print("Participant:", i, " Camera", cam, " joint", maps[j])
                    #print("MAX", c, "Index", l)
                    #print("___________________________________________________________")



In [ ]:
df = pd.DataFrame(data_2)
df

In [ ]:
df[df['Validation_set'] == True]